# 03a - RM-a: full fine-tuning IndoBERT

Baseline penelitian. Seluruh 109 juta parameter IndoBERT diperbarui, sehingga
skenario ini yang paling mahal sekaligus menjadi pembanding performa untuk dua
strategi ringan.

Notebook ini menjalankan KAMPANYE PENUH RM-a: kalibrasi biaya, lalu seluruh
grid di `tuning_grids/RMA_TUNING_GRID*.csv` (grid kombinatorial `lr x epochs x
batch`, coordinate descent `warmup_ratio` dan `weight_decay`). Keluaran ditulis
ke `outputs/tuning/`, folder yang sama yang dipakai bersama oleh
`03b_rmb_frozen.ipynb`, `03c_rmc_rac.ipynb`, dan `05_final_benchmark.ipynb`.

Prinsipnya human-in-the-loop: tidak ada pencarian otomatis, urutan konfigurasi
ditentukan manusia, dan setiap baris hasil membawa kolom `catatan` yang
merekam alasan konfigurasi itu dicoba. Seleksi memakai split validation; split
test tidak disentuh sama sekali di notebook ini.

Prasyarat: `02_preprocessing.ipynb` sudah dijalankan.


In [ ]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir

runner = CampaignRunner(out_dir=OUT_DIR)
hardware = runner.write_hardware()
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])


## 1. Konteks hardware

In [ ]:
for key, value in hardware.items():
    print(f"  {key:16s}: {value}")


`hardware.json` mencatat GPU, VRAM, driver, CUDA, CPU, RAM, OS, versi Python,
torch, transformers, faiss, seed, serta waktu catat dan waktu boot mesin tiap sesi
tuning. Bila folder ini sudah memuat hasil dari lingkungan lain, sel di atas
berhenti dengan error: angka efisiensi hanya sah dari satu hardware (Aturan #1).
Melanjutkan kampanye setelah mesin di-restart menambah sesi baru, bukan menimpa.

`05_final_benchmark.ipynb` memeriksa ulang lingkungan ini dan berhenti bila berbeda.


## 2. Kalibrasi biaya

Jalankan satu konfigurasi lebih dulu untuk mengukur waktu dan memori
sesungguhnya di mesin ini, sebelum mempertaruhkan berjam-jam pada grid penuh.
Kalau memori kurang, turunkan `MICRO_BATCH` di `.env`; batch efektif tidak
berubah karena selisihnya ditutup akumulasi gradien. Konfigurasi ini sekaligus
menjadi run pertama grid, bukan sekadar uji coba yang dibuang.


In [ ]:
kalibrasi = runner.run(
    "rma",
    {"lr": 2e-5, "epochs": 5, "batch": 16, "warmup_ratio": 0.1, "weight_decay": 0.01},
    note="kalibrasi biaya: baseline kanonik IndoNLU/Wilie 2020, sekaligus run #1 grid",
)

per_run = kalibrasi["train_time_s"]
print(f"run #{kalibrasi['run_id']}")
print(f"  val F1-macro    : {kalibrasi['val_f1_macro']:.4f}")
print(f"  waktu satu run  : {per_run:.0f} s")
print(f"  peak GPU memory : {kalibrasi['peak_mem_mb']:.0f} MB")


Akumulasi gradien membuat RUMUS gradiennya ekuivalen dengan batch besar (BERT
memakai LayerNorm, bukan BatchNorm, dan loss dibagi jumlah akumulasi), tetapi
RUN-nya tidak identik: `DataLoader` dengan ukuran batch berbeda mengonsumsi RNG
secara berbeda, sehingga mask dropout dan komposisi tiap batch ikut berubah.

Terukur di kampanye ini: `micro_batch` efektif 8 versus 16 pada konfigurasi yang
sama persis memberi val F1-macro 0,974873 versus 0,977266. Karena itu
`micro_batch` harus dikunci untuk seluruh sel satu grid dan disebutkan di Bab 4
sebagai bagian konfigurasi, bukan diperlakukan sebagai knob memori bebas.


## 3. Muat rancangan grid

In [ ]:
import pandas as pd

GRID_DIR = settings.data_dir.parent / "tuning_grids"
RMA_GRIDS = ("RMA_TUNING_GRID.csv", "RMA_TUNING_GRID_STAGE2.csv")


def muat_grid(nama: str) -> list[dict]:
    frame = pd.read_csv(GRID_DIR / nama)
    catatan = frame.pop("catatan") if "catatan" in frame.columns else ""
    return [
        {"config": {k: v for k, v in baris.items() if pd.notna(v)},
         "note": catatan.iloc[i] if hasattr(catatan, "iloc") else ""}
        for i, baris in enumerate(frame.to_dict("records"))
    ]


for berkas in RMA_GRIDS:
    print(f"  {berkas}: {len(pd.read_csv(GRID_DIR / berkas))} konfigurasi")


## Melanjutkan kampanye yang terputus

`run_batch` menyaring konfigurasi yang sudah ada di riwayat secara default
(`resume=True`), sehingga sel batch di bawah aman dijalankan ulang apa adanya
setelah kernel mati, listrik padam, atau proses dihentikan. Yang sudah selesai
dilewati, penomoran run berlanjut, dan `best.json` tetap terjaga. Perbandingan
memakai konfigurasi LENGKAP setelah nilai default diisi, dan `micro_batch`
dinormalkan ke nilai efektifnya (`min(batch, micro_batch)`).


In [ ]:
permintaan_rma = [item for berkas in RMA_GRIDS for item in muat_grid(berkas)]
tersisa = runner.pending_requests("rma", permintaan_rma)
print(f"{len(permintaan_rma) - len(tersisa)}/{len(permintaan_rma)} selesai, {len(tersisa)} tersisa")


## 4. Grid tahap 1: lr x epochs x batch

Ketiganya digrid bersama karena bersama-sama menentukan lintasan optimasi
(batch 32 pada 5 epoch memberi separuh jumlah langkah pembaruan dibanding
batch 16). Konfigurasi yang gagal diisolasi ke `runs_rma_errors.csv` dan tidak
menghentikan sisa antrean.


In [ ]:
runner.run_batch("rma", muat_grid("RMA_TUNING_GRID.csv"), batch_id="rma_tahap1_grid")

# Dibaca dari riwayat, bukan dari nilai kembalian run_batch: bila seluruh konfigurasi
# sudah tercatat (kampanye dilanjutkan), run_batch mengembalikan tabel kosong.
riwayat_rma = runner.reporter.runs_frame("rma")
riwayat_rma[riwayat_rma["batch_id"] == "rma_tahap1_grid"].nlargest(10, "val_f1_macro")[
    ["run_id", "lr", "epochs", "batch", "val_f1_macro", "val_f1_judi",
     "train_time_s", "is_tie_with_best"]
]


Baca grid sebagai permukaan, bukan daftar. Heatmap `lr x epochs` per nilai
`batch` di `outputs/tuning/figures/` memperlihatkan apakah learning rate optimal
ikut bergeser saat batch berubah. Pemenang yang duduk di tepi grid adalah sinyal
untuk melebarkan rentang, bukan untuk langsung dikunci. Selisih di bawah 0,15 pp
dihitung seri karena hanya ada satu seed; pada kondisi seri, pilih konfigurasi
yang lebih murah.


## 5. Coordinate descent: warmup_ratio dan weight_decay

Efektif independen dari grup A (`lr x epochs x batch`) sehingga cukup dicoba
satu per satu di sel pemenang, bukan digrid bersama.


In [ ]:
runner.run_batch("rma", muat_grid("RMA_TUNING_GRID_STAGE2.csv"),
                 batch_id="rma_tahap2_coordinate")

riwayat_rma = runner.reporter.runs_frame("rma")
riwayat_rma[riwayat_rma["batch_id"] == "rma_tahap2_coordinate"][
    ["run_id", "warmup_ratio", "weight_decay", "val_f1_macro",
     "delta_vs_best_f1_macro_pp", "is_tie_with_best"]
]


## 6. Kurva epoch juara

In [ ]:
import json

best_rma = json.loads((OUT_DIR / "best.json").read_text(encoding="utf-8"))["rma"]
print(f"juara RM-a: run #{best_rma['run_id']} | val F1-macro {best_rma['val_f1_macro']:.4f}")

history = pd.read_csv(OUT_DIR / "history" / "rma_history.csv")
history[history["run_id"] == best_rma["run_id"]]


`overfit_signal` di baris riwayat bernilai True bila epoch terbaik bukan epoch
terakhir, yaitu tanda bahwa menambah epoch justru memperburuk validasi. Figur
kurva dan confusion matrix juara tersimpan di `outputs/tuning/figures/`.


## Ringkasan

Juara RM-a tercatat di `best.json`, checkpoint-nya di `checkpoints/rma_best.pt`.
Checkpoint tiga run teratas (F1-macro, lalu F1 judi, lalu training time) disimpan
bergulir di `checkpoints/rma_top/`, supaya kandidat #2 bisa dievaluasi di test
tanpa pelatihan ulang. Angka di atas berasal dari split validation; split test
masih tertutup dan baru dibuka satu kali di `05_final_benchmark.ipynb`.

Kalau ingin menambah konfigurasi setelah membaca hasil, panggil `runner.run`
atau `runner.run_batch` lagi: riwayat menumpuk dan penomoran run berlanjut.

Lanjut ke `03b_rmb_frozen.ipynb`.
